# [ Computer Vision ] [ CSE445 ] [ Assignment-2 ]

**Dataset Name:** IoTKITs<br>
**Source Link:** https://data.mendeley.com/datasets/x5thzmkxhy/1

## Configuration

In [1]:
!pip install ultralytics --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.6 MB/s eta 0:00:00


In [2]:
import json
import random
import shutil
import hashlib
import os

dataset_dir = "/kaggle/input/datasets/Mrpaul0007/iotkits"
working_dir = "/kaggle/working/"

split_dir   = os.path.join(working_dir, "splits")
os.makedirs(split_dir, exist_ok=True)

train_dir = os.path.join(dataset_dir, "train")
valid_dir = os.path.join(dataset_dir, "valid")
test_dir  = os.path.join(dataset_dir, "test")

train_annotation = os.path.join(train_dir, "_annotations.coco.json")
valid_annotation = os.path.join(valid_dir, "_annotations.coco.json")
test_annotation  = os.path.join(test_dir, "_annotations.coco.json")

role_of_folder = {
    "train": "ssl_pool",
    "valid": "val",
    "test":  "test",
}

SEED = 42
RHO = 0.20
IMGSZ = 640

print("====== Configuration ======\n")
print("Dataset path      :", dataset_dir)
print("Train path        :", train_dir)
print("Valid path        :", valid_dir)
print("Test path         :", test_dir)
print("Output split path :", split_dir,"\n")

print("Role Mapping:")
for folder in role_of_folder:
    print(folder, "->", role_of_folder[folder],)

print("\nSeed              :", SEED)
print("Label fraction rho:", RHO,)
print("Image size:", IMGSZ)
print("Configuration loaded successfully!")

====== Configuration ======

Dataset path      : /kaggle/input/datasets/Mrpaul0007/iotkits
Train path        : /kaggle/input/datasets/Mrpaul0007/iotkits/train
Valid path        : /kaggle/input/datasets/Mrpaul0007/iotkits/valid
Test path         : /kaggle/input/datasets/Mrpaul0007/iotkits/test
Output split path : /kaggle/working/splits 

Role Mapping:
train -> ssl_pool
valid -> val
test -> test

Seed              : 42
Label fraction rho: 0.2
Image size: 640
Configuration loaded successfully!


## Loading Part A Splits

In [3]:
def load_coco(path):
    with open(path, "r") as f:
        return json.load(f)

# containers
image_info   = {}
annotations  = {}
folder_of    = {}
cats         = {}

folder_paths = {
    "train": (train_dir, train_annotation),
    "valid": (valid_dir, valid_annotation),
    "test":  (test_dir,  test_annotation),
}

for folder in folder_paths:
    directory, annotation_path = folder_paths[folder]
    coco = load_coco(annotation_path)

    # class names
    for c in coco["categories"]:
        if c["id"] != 0:
            cats[c["id"]] = c["name"]

    # images
    old_to_new_id = {}
    for img in coco["images"]:
        uid = folder + "_" + str(img["id"])
        old_to_new_id[img["id"]] = uid

        image_info[uid] = {
            "file_name": img["file_name"],
            "width":     img["width"],
            "height":    img["height"],
            "src_dir":   directory,
        }
        annotations[uid] = []
        folder_of[uid] = folder

    # annotations
    for a in coco["annotations"]:
        uid = old_to_new_id[a["image_id"]]
        annotations[uid].append({
            "category_id": a["category_id"],
            "bbox":        a["bbox"],
        })

    n_img = len(coco["images"])
    n_ann = len(coco["annotations"])
    print(f"{folder:<6} : {n_img:>5} images, {n_ann:>5} annotations")

print()
print("Total images        :", len(image_info))
print("Total classes       :", len(cats))

train  :  2485 images,  2619 annotations
valid  :   311 images,   322 annotations
test   :   311 images,   325 annotations

Total images        : 3107
Total classes       : 32


## Assigning Part B Partitions

In [4]:
# Dropping Zero-Annotation
kept_uids = []

before = len(image_info)
for uid in image_info:
    if len(annotations[uid]) > 0:
        kept_uids.append(uid)
removed = before - len(kept_uids)

ssl_pool = []
val      = []
test     = []

for uid in kept_uids:
    folder = folder_of[uid]
    role = role_of_folder[folder]

    if role == "ssl_pool":
        ssl_pool.append(uid)
    elif role == "val":
        val.append(uid)
    elif role == "test":
        test.append(uid)

ssl_pool.sort()
val.sort()
test.sort()

N = len(ssl_pool) + len(val) + len(test)

print("Number of Images [before] :", before)
print("Removed [zero annotations]:", removed)
print("Number of Images [after]  :", len(kept_uids))
print("SSL pool                  :", len(ssl_pool))
print("Validation                :", len(val))
print("Test                      :", len(test))
print("|D|                       :", N)

Number of Images [before] : 3107
Removed [zero annotations]: 0
Number of Images [after]  : 3107
SSL pool                  : 2485
Validation                : 311
Test                      : 311
|D|                       : 3107


## Fine-Tune Subset

In [5]:
shuffled_pool = list(ssl_pool)
shuffled_pool.sort()

rng = random.Random(SEED)
rng.shuffle(shuffled_pool)

n_labelled = int(round(RHO * N))
labelled = shuffled_pool[:n_labelled]
labelled.sort()

unused = len(ssl_pool) - len(labelled)

print("Target subset size (20% of |D|):", n_labelled)
print("Actual subset size             :", len(labelled))
print("Share of the SSL pool          :", round(100 * len(labelled) / len(ssl_pool), 1), "%")
print("Pool images left unlabelled    :", unused)

Target subset size (20% of |D|): 621
Actual subset size             : 621
Share of the SSL pool          : 25.0 %
Pool images left unlabelled    : 1864


## Partition Sizes

In [6]:
def count_instances(uid_list):
    total = 0
    for uid in uid_list:
        total = total + len(annotations[uid])
    return total

def count_classes(uid_list):
    seen = set()
    for uid in uid_list:
        for a in annotations[uid]:
            seen.add(a["category_id"])
    return len(seen)

rows = [
    ("test holdout",          test),
    ("validation",            val),
    ("SSL pool (labels off)", ssl_pool),
    ("labelled D_0.20",       labelled),
]


print(f"PARTITION   |D| = {N} images, seed = {SEED}\n")
print(f"{'split':<24}{'imgs':>7}{'% of D':>9}{'ann':>8}{'cls':>6}")

for name, uid_list in rows:
    share = 100 * len(uid_list) / N
    print(f"{name:<24}{len(uid_list):>7}{share:>8.1f}%"
          f"{count_instances(uid_list):>8}{count_classes(uid_list):>6}")

print(f"\nUnlabelled, unused in fine-tuning: {unused} "
      f"({round(100 * unused / N, 1)}% of |D|)")


PARTITION   |D| = 3107 images, seed = 42

split                      imgs   % of D     ann   cls
test holdout                311    10.0%     325    32
validation                  311    10.0%     322    32
SSL pool (labels off)      2485    80.0%    2619    32
labelled D_0.20             621    20.0%     660    32

Unlabelled, unused in fine-tuning: 1864 (60.0% of |D|)


## Leakage Verification

In [7]:
def filenames_of(uid_list):
    names = set()
    for uid in uid_list:
        names.add(image_info[uid]["file_name"])
    return names

test_names = filenames_of(test)
val_names  = filenames_of(val)
pool_names = filenames_of(ssl_pool)

checks = [
    ("test  ∩ ssl_pool", test_names & pool_names),
    ("val   ∩ ssl_pool", val_names  & pool_names),
    ("test  ∩ val",      test_names & val_names),
]


print("DISJOINTNESS CHECKS")

# Verify Overlap by file_name (In "Assignment-1" we checked it through ids)
for name, overlap in checks:
    status = "PASS" if len(overlap) == 0 else "FAIL"
    print(f"{name:<20} {len(overlap):>4}   {status}")

for name, overlap in checks:
    assert len(overlap) == 0, "Leakage detected in: " + name

# Verify Subset
assert set(labelled).issubset(set(ssl_pool)), "Labelled subset leaves the SSL pool!"
print(f"{'D 0.20 ⊆ ssl_pool':<20} {'':>4}   PASS")

# Verify D
assert len(ssl_pool) + len(val) + len(test) == N
print(f"{'partitions cover D':<20} {'':>4}   PASS")

print("All verification checks passed.")

DISJOINTNESS CHECKS
test  ∩ ssl_pool        0   PASS
val   ∩ ssl_pool        0   PASS
test  ∩ val             0   PASS
D 0.20 ⊆ ssl_pool           PASS
partitions cover D          PASS
All verification checks passed.


## Save The Splits

In [8]:
def save_split(file_name, uid_list):
    lines = []
    for uid in uid_list:
        info = image_info[uid]
        path = info["src_dir"] + "/" + info["file_name"]
        lines.append(path)
    out_path = split_dir + "/" + file_name
    f = open(out_path, "w")
    for line in lines:
        f.write(line + "\n")
    f.close()
    print("saved", len(uid_list), "paths in",  file_name)


print("Saving split files to:", split_dir)
save_split("test.txt", test)
save_split("val.txt", val)
save_split("ssl_pool.txt", ssl_pool)
save_split("labelled_20.txt", labelled)

# Create Manifest
class_dict = {}
for k in cats:
    class_dict[str(k)] = cats[k]

manifest = {}
manifest["seed"] = SEED
manifest["rho"] = RHO
manifest["imgsz"] = IMGSZ
manifest["n_total"] = N
manifest["source"] = "In Assignment-1, we merged the original train and validation datasets and then re-split the combined dataset into train, validation, and test sets using a shared random seed of 42."
manifest["Image counts"] = {
    "test": len(test),
    "val": len(val),
    "ssl_pool": len(ssl_pool),
    "labelled_20": len(labelled)
}
manifest["Annotations"] = {
    "test": count_instances(test),
    "val": count_instances(val),
    "ssl_pool": count_instances(ssl_pool),
    "labelled_20": count_instances(labelled)
}
manifest["classes"] = class_dict

manifest_path = split_dir + "/manifest.json"
f = open(manifest_path, "w")
json.dump(manifest, f, indent=4)
f.close()
print("Manifest saved:", manifest_path)

Saving split files to: /kaggle/working/splits
saved 311 paths in test.txt
saved 311 paths in val.txt
saved 2485 paths in ssl_pool.txt
saved 621 paths in labelled_20.txt
Manifest saved: /kaggle/working/splits/manifest.json


## Coco To Yolo Format

In [9]:
class_names = []
cat_to_yolo = {}

for yolo_id, category_id in enumerate(sorted(cats)):
    cat_to_yolo[category_id] = yolo_id
    class_names.append(cats[category_id])
print("Successfully Done Mapping!")


def coco_box_to_yolo_line(x, y, w, h, img_w, img_h, class_id):
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    nw = w / img_w
    nh = h / img_h
    return f"{class_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"


def save_yolo_split(uid_list, split_name, yolo_root):
    image_out = os.path.join(yolo_root, "images", split_name)
    label_out = os.path.join(yolo_root, "labels", split_name)
    os.makedirs(image_out, exist_ok=True)
    os.makedirs(label_out, exist_ok=True)

    for uid in uid_list:
        info = image_info[uid]
        img_w, img_h = info["width"], info["height"]

        # add uid before filename for never clash
        base, ext = os.path.splitext(info["file_name"])
        stem = uid + "__" + base

        src_path = os.path.join(info["src_dir"], info["file_name"])
        dst_path = os.path.join(image_out, stem + ext)
        shutil.copy(src_path, dst_path)

        lines = []
        for a in annotations[uid]:
            x, y, w, h = a["bbox"]
            if w <= 0 or h <= 0:
                continue
            class_id = cat_to_yolo[a["category_id"]]
            lines.append(coco_box_to_yolo_line(x, y, w, h, img_w, img_h, class_id))

        label_path = os.path.join(label_out, stem + ".txt")
        with open(label_path, "w") as f:
            if lines:
                f.write("\n".join(lines))
                f.write("\n")

    print(f"  {split_name:<6} {len(uid_list):>5} images written")
yolo_root = os.path.join(working_dir, "yolo_rho20")

if os.path.exists(yolo_root):
    shutil.rmtree(yolo_root)

# for labeled
yolo_splits = {
    "train": labelled,
    "val":   val,
    "test":  test,
}

for split_name, uid_list in yolo_splits.items():
    save_yolo_split(uid_list, split_name, yolo_root)


# data.yaml
yaml_text = (
    f"path: {yolo_root}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"nc: {len(class_names)}\n"
    f"names: {json.dumps(class_names)}\n"
)

yaml_path = os.path.join(yolo_root, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_text)

print()
print("yaml path:", yaml_path)
print()
print(yaml_text)

Successfully Done Mapping!
  train    621 images written
  val      311 images written
  test     311 images written

yaml path: /kaggle/working/yolo_rho20/data.yaml

path: /kaggle/working/yolo_rho20
train: images/train
val: images/val
test: images/test
nc: 32
names: ["Arduino Mega 2560 -Black-", "Arduino Mega 2560 -Blue-", "Arduino Mega 2560 -Yellow-", "Arduino Uno -Black-", "Arduino Uno -Green-", "Arduino Uno Camera Shield", "Arduino Uno WiFi Shield", "Arduino-Due", "Arduino-Micro", "Arduino-Nano", "Arduino-ProMini", "Arduino-Zero", "Arduino-leonardo", "ESP32", "Esp8266", "Jetson Nano", "Jetson TX2", "Raspberry Pi 1 B-", "Raspberry Pi 3 B-", "Raspberry-Pi-1-A-", "Raspberry-Pi-2-B", "Raspberry-Pi-3-A-", "Raspberry-Pi-3-B", "Raspberry-Pi-4-B", "Raspberry-Pi-5", "Raspberry-Pi-Zero", "Raspberry-Pi-Zero-2-W", "Raspberry-Pi-Zero-W", "Raspberry-Pi-Zero-WH", "STM32", "TelosB", "Wemos"]



## Unlabelled SSL Pool Folder

In [10]:
ssl_image_dir = os.path.join(working_dir, "ssl_pool_images")

if os.path.exists(ssl_image_dir):
    shutil.rmtree(ssl_image_dir)
os.makedirs(ssl_image_dir)

img_count = 0

for uid in ssl_pool:
    info = image_info[uid]
    file_name = info["file_name"]

    # add uid before filename for never clash
    base, ext = os.path.splitext(file_name)
    stem = uid + "__" + base

    src_path = os.path.join(info["src_dir"], file_name)
    dst_path = os.path.join(ssl_image_dir, stem + ext)
    shutil.copy(src_path, dst_path)
    img_count = img_count + 1

print("SSL pool image folder :", ssl_image_dir)
print("Images                :", img_count)
print("Label files           : 0  (annotations discarded by design)")

SSL pool image folder : /kaggle/working/ssl_pool_images
Images                : 2485
Label files           : 0  (annotations discarded by design)


## Final Leakage Verification

In [11]:
def original_names(folder):
    names = set()
    for file_name in os.listdir(folder):
        stem = file_name.split("__", 1)[1]
        names.add(stem)
    return names

ssl_names   = original_names(ssl_image_dir)
train_names = original_names(os.path.join(yolo_root, "images", "train"))
val_names   = original_names(os.path.join(yolo_root, "images", "val"))
test_names  = original_names(os.path.join(yolo_root, "images", "test"))

checks = [
    ("val   ∩ ssl_pool", val_names  & ssl_names),
    ("test  ∩ ssl_pool", test_names & ssl_names),
    ("val   ∩ test",     val_names  & test_names),
]

print("LEAKAGE CHECK")
print("-"*15)
for name, overlap in checks:
    status = "OK" if len(overlap) == 0 else "LEAK!"
    print(f"  {name:<20} {len(overlap):>4}   {status}")
    assert len(overlap) == 0, f"Leakage detected: {name}"

not_in_pool = train_names - ssl_names
status = "PASS" if len(not_in_pool) == 0 else "PROBLEM"
print(f"  {'train not in pool':<20} {len(not_in_pool):>4}   {status}")
assert len(not_in_pool) == 0, "Train contains images outside the SSL pool!"
print("No leakage found across ssl_pool_images / val / test.")
print("train is confirmed to be a subset of the ssl pool, as intended.")

LEAKAGE CHECK
---------------
  val   ∩ ssl_pool        0   OK
  test  ∩ ssl_pool        0   OK
  val   ∩ test            0   OK
  train not in pool       0   PASS
No leakage found across ssl_pool_images / val / test.
train is confirmed to be a subset of the ssl pool, as intended.
